# Baseline Default: NTO AI 25/26 Final (Lost Positive Interactions)

## Что получится после запуска

1. Локальная валидация с диагностиками покрытия кандидатов.
2. Обученный `CatBoostClassifier` pointwise на masking-сценарии.
3. `submission.csv`, валидный по контракту (`20` уникальных `edition_id` на пользователя).
4. Артефакты запуска в `outputs/baseline_default/` + копия сабмита в корне проекта.

## Checkpoint policy

- Каждый тяжёлый этап сохраняет промежуточные артефакты в `outputs/baseline_default/checkpoints/<stage>`.
- Если cache валиден (hash входов/конфига совпадает), этап автоматически загружается (`CACHE HIT`) и не пересчитывается.
- Если cache устарел или неполный — выполняется пересчёт и перезапись (`CACHE SAVE`).
- Для принудительного полного пересчёта установите `cfg.force_recompute = True`.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import random
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from tqdm.auto import tqdm

TOP_K = 20


@dataclass
class Config:
    data_dir: Path = Path("data")
    output_dir: Path = Path("outputs/baseline_default")
    checkpoint_dir: Path = Path("outputs/baseline_default/checkpoints")
    root_submission_path: Path = Path("submission.csv")

    enable_checkpoints: bool = True
    force_recompute: bool = False
    checkpoint_version: str = "v1"

    seed: int = 42

    history_window_days: int = 150
    incident_window_days: int = 30
    mask_ratio: float = 0.2

    top_k: int = 20
    min_candidates_per_user: int = 80

    pop_top_n_per_window: int = 60
    content_top_n: int = 80
    i2i_top_n: int = 80

    content_recent_k: int = 15
    i2i_recent_k_for_stats: int = 10
    i2i_anchor_k: int = 10
    i2i_neighbors_per_item: int = 80

    catboost_iterations: int = 350
    catboost_depth: int = 6
    catboost_learning_rate: float = 0.08
    catboost_l2_leaf_reg: float = 4.0


cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
cfg.checkpoint_dir.mkdir(parents=True, exist_ok=True)
print("Config loaded")
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}, ensure_ascii=False, indent=2))

Config loaded
{
  "data_dir": "data",
  "output_dir": "outputs/baseline_default",
  "checkpoint_dir": "outputs/baseline_default/checkpoints",
  "root_submission_path": "submission.csv",
  "enable_checkpoints": true,
  "force_recompute": false,
  "checkpoint_version": "v1",
  "seed": 42,
  "history_window_days": 150,
  "incident_window_days": 30,
  "mask_ratio": 0.2,
  "top_k": 20,
  "min_candidates_per_user": 80,
  "pop_top_n_per_window": 60,
  "content_top_n": 80,
  "i2i_top_n": 80,
  "content_recent_k": 15,
  "i2i_recent_k_for_stats": 10,
  "i2i_anchor_k": 10,
  "i2i_neighbors_per_item": 80,
  "catboost_iterations": 350,
  "catboost_depth": 6,
  "catboost_learning_rate": 0.08,
  "catboost_l2_leaf_reg": 4.0
}


In [2]:
# ============================================================
# Global constants used across all notebook sections.
# Put new column names and artifact keys here first.
# ============================================================

# Column names
COL_USER = "user_id"
COL_ITEM = "edition_id"
COL_EVENT_TYPE = "event_type"
COL_EVENT_TS = "event_ts"
COL_RATING = "rating"
COL_RANK = "rank"
COL_SCORE = "score"
COL_LABEL = "label"
COL_SOURCE = "candidate_source"
COL_SOURCE_SCORE = "candidate_source_score"

COL_BOOK_ID = "book_id"
COL_AUTHOR_ID = "author_id"
COL_PUBLICATION_YEAR = "publication_year"
COL_LANGUAGE_ID = "language_id"
COL_PUBLISHER_ID = "publisher_id"
COL_GENDER = "gender"
COL_AGE = "age"
COL_GENRE_ID = "genre_id"
COL_AGE_RESTRICTION = "age_restriction"

# Dataset keys
DATA_INTERACTIONS = "interactions"
DATA_EDITIONS = "editions"
DATA_BOOK_GENRES = "book_genres"
DATA_USERS = "users"
DATA_TARGETS = "targets"
DATA_AUTHORS = "authors"
DATA_GENRES = "genres"

# Feature names (commonly reused)
F_ITEM_POP_180D = "f_item_pop_users_180d"
F_UI_LANG_SEEN = "f_ui_same_language_seen_before"
F_UI_PUB_SEEN = "f_ui_same_publisher_seen_before"

# Common values
POSITIVE_EVENT_TYPES = (1, 2)
POPULARITY_WINDOWS = (7, 14, 30)
JOIN_LEFT = "left"

# Artifact names
ARTIFACT_SUBMISSION = "submission.csv"
ARTIFACT_METRICS = "metrics.json"
ARTIFACT_CONFIG = "run_config.json"

# Checkpoint layout
CHECKPOINT_META_FILE = "meta.json"
CHECKPOINT_LOCAL_METRICS_FILE = "local_metrics.json"
CHECKPOINT_MODEL_FILE = "model.cbm"

STAGE_RAW_DATA = "raw_data"
STAGE_PREPROCESS_MASKING = "preprocess_masking"
STAGE_LOCAL_CANDIDATES = "local_candidates"
STAGE_LOCAL_FEATURES_TRAIN = "local_features_train"
STAGE_FINAL_CANDIDATES = "final_candidates"
STAGE_FINAL_FEATURES_SCORES_SUBMISSION = "final_features_scores_submission"

checkpoint_summary: list[dict[str, Any]] = []


def checkpoint_status_print(stage: str, status: str, details: str = "") -> None:
    """Print and store checkpoint status for final run summary."""
    msg = f"[{status}] {stage}"
    if details:
        msg += f" :: {details}"
    print(msg)
    checkpoint_summary.append({"stage": stage, "status": status, "details": details})


def stage_dir(stage: str) -> Path:
    return cfg.checkpoint_dir / stage


def compute_hash(payload: Any) -> str:
    """Build deterministic short hash for metadata/config structures."""
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


def file_fingerprint(path: Path) -> dict[str, Any]:
    """Capture source file modification footprint for cache invalidation."""
    stat = path.stat()
    return {
        "path": str(path),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def stage_expected_hash(stage: str, payload: dict[str, Any]) -> str:
    return compute_hash({
        "stage": stage,
        "checkpoint_version": cfg.checkpoint_version,
        "payload": payload,
    })


def write_json(path: Path, payload: dict[str, Any]) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_stage_df_dict(stage: str, frames: dict[str, pd.DataFrame], expected_hash: str, payload: dict[str, Any]) -> None:
    """Persist a dataframe bundle and its validation metadata."""
    sdir = stage_dir(stage)
    sdir.mkdir(parents=True, exist_ok=True)

    for key, df in frames.items():
        df.to_parquet(sdir / f"{key}.parquet", index=False)

    meta = {
        "stage": stage,
        "checkpoint_version": cfg.checkpoint_version,
        "expected_hash": expected_hash,
        "payload": payload,
        "files": sorted([f"{k}.parquet" for k in frames.keys()]),
        "saved_at": time.time(),
    }
    write_json(sdir / CHECKPOINT_META_FILE, meta)


def load_stage_df_dict(stage: str, expected_hash: str, required_keys: list[str]) -> tuple[dict[str, pd.DataFrame] | None, dict[str, Any] | None]:
    """Load a dataframe checkpoint only if hash and file list are valid."""
    if cfg.force_recompute or not cfg.enable_checkpoints:
        return None, None

    sdir = stage_dir(stage)
    meta_path = sdir / CHECKPOINT_META_FILE
    if not meta_path.exists():
        return None, None

    meta = read_json(meta_path)
    if meta.get("checkpoint_version") != cfg.checkpoint_version:
        return None, None
    if meta.get("expected_hash") != expected_hash:
        return None, None

    missing = [k for k in required_keys if not (sdir / f"{k}.parquet").exists()]
    if missing:
        return None, None

    frames = {k: pd.read_parquet(sdir / f"{k}.parquet") for k in required_keys}
    return frames, meta


def save_stage_model(stage: str, ranker: "BaselineRanker", expected_hash: str, payload: dict[str, Any]) -> None:
    """Persist trained CatBoost model with metadata for reload."""
    sdir = stage_dir(stage)
    sdir.mkdir(parents=True, exist_ok=True)

    model_path = sdir / CHECKPOINT_MODEL_FILE
    ranker.model.save_model(str(model_path))

    model_meta = {
        "feature_cols": ranker.feature_cols,
        "expected_hash": expected_hash,
        "checkpoint_version": cfg.checkpoint_version,
        "payload": payload,
        "saved_at": time.time(),
    }
    write_json(sdir / "model_meta.json", model_meta)


def load_stage_model(stage: str, expected_hash: str) -> dict[str, Any] | None:
    """Load model metadata when a compatible checkpoint exists."""
    if cfg.force_recompute or not cfg.enable_checkpoints:
        return None

    sdir = stage_dir(stage)
    meta_path = sdir / "model_meta.json"
    model_path = sdir / CHECKPOINT_MODEL_FILE
    if not meta_path.exists() or not model_path.exists():
        return None

    meta = read_json(meta_path)
    if meta.get("checkpoint_version") != cfg.checkpoint_version:
        return None
    if meta.get("expected_hash") != expected_hash:
        return None
    return {"meta": meta, "model_path": model_path}


def set_seed(seed: int) -> None:
    """Fix random state to keep local experiments reproducible."""
    random.seed(seed)
    np.random.seed(seed)


def print_step(title: str) -> None:
    """Print a visible section separator in notebook output."""
    print(f"\n{'=' * 12} {title} {'=' * 12}")


set_seed(cfg.seed)
print("Seed fixed:", cfg.seed)

Seed fixed: 42


## Раздел кода 1 (Data Loading + Preprocessing + Masking)

- `DataLoader` — читает и валидирует входные таблицы
- `Preprocessor` — готовит наблюдаемые позитивные взаимодействия
- `MaskingValidator` — строит локальный masking-сценарий для offline-оценки

### Контракт: `DataLoader`

**Зачем:** централизовать чтение/валидацию сырых таблиц до старта любой логики ML.  
**Вход:** `Config.data_dir` и обязательные CSV.  
**Выход:** `dict[str, DataFrame]` с ключами датасетов.  
**Риски:** если схема изменилась, шаг падает сразу с понятной ошибкой.


In [3]:
class DataLoader:
    """Read raw CSV files and validate schema before any feature logic.

    Keeping schema checks here prevents hidden downstream errors and
    makes onboarding easier: participants fail fast with clear messages.
    """

    REQUIRED_FILES = {
        DATA_INTERACTIONS: "interactions.csv",
        DATA_TARGETS: "targets.csv",
        DATA_EDITIONS: "editions.csv",
        DATA_BOOK_GENRES: "book_genres.csv",
        DATA_USERS: "users.csv",
    }

    REQUIRED_COLUMNS = {
        DATA_INTERACTIONS: {COL_USER, COL_ITEM, COL_EVENT_TYPE, COL_RATING, COL_EVENT_TS},
        DATA_TARGETS: {COL_USER},
        DATA_EDITIONS: {COL_ITEM, "book_id", "author_id", "publication_year", "age_restriction", "language_id", "publisher_id"},
        DATA_BOOK_GENRES: {"book_id", "genre_id"},
        DATA_USERS: {COL_USER, "gender", "age"},
    }

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def load(self) -> dict[str, pd.DataFrame]:
        data = {}
        for key, file_name in self.REQUIRED_FILES.items():
            path = self.cfg.data_dir / file_name
            if not path.exists():
                raise FileNotFoundError(f"Missing file: {path}")
            data[key] = pd.read_csv(path)
        self._validate(data)
        return data

    def _validate(self, data: dict[str, pd.DataFrame]) -> None:
        for key, required in self.REQUIRED_COLUMNS.items():
            missing = required - set(data[key].columns)
            if missing:
                raise ValueError(f"{key}: missing columns {sorted(missing)}")

        interactions = data[DATA_INTERACTIONS]
        bad_event_types = set(interactions[COL_EVENT_TYPE].dropna().unique()) - {1, 2}
        if bad_event_types:
            raise ValueError(f"Unexpected event_type values: {bad_event_types}")


class Preprocessor:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def preprocess_interactions(self, interactions: pd.DataFrame) -> pd.DataFrame:
        df = interactions.copy()
        df = df.drop_duplicates()
        df[COL_EVENT_TS] = pd.to_datetime(df[COL_EVENT_TS], errors="coerce")
        if df[COL_EVENT_TS].isna().any():
            raise ValueError("event_ts contains unparsable values")
        df = df[df[COL_EVENT_TYPE].isin([1, 2])].copy()
        df = df.sort_values([COL_USER, COL_EVENT_TS, COL_ITEM]).reset_index(drop=True)

        min_ts = df[COL_EVENT_TS].min()
        max_ts = df[COL_EVENT_TS].max()
        incident_start = max_ts.normalize() - pd.Timedelta(days=self.cfg.incident_window_days - 1)

        df["event_date"] = df[COL_EVENT_TS].dt.date
        df["day_idx"] = (df[COL_EVENT_TS].dt.normalize() - min_ts.normalize()).dt.days
        df["is_incident_window"] = (df[COL_EVENT_TS].dt.normalize() >= incident_start).astype(int)
        df["is_positive"] = 1
        return df

    def build_pair_table(self, interactions: pd.DataFrame) -> pd.DataFrame:
        g = interactions.groupby([COL_USER, COL_ITEM], as_index=False)
        pair_df = g.agg(
            first_ts=(COL_EVENT_TS, "min"),
            last_ts=(COL_EVENT_TS, "max"),
            cnt_events=(COL_EVENT_TS, "size"),
            cnt_wishlist=(COL_EVENT_TYPE, lambda s: int((s == 1).sum())),
            cnt_read=(COL_EVENT_TYPE, lambda s: int((s == 2).sum())),
            mean_rating_read=(COL_RATING, "mean"),
        )
        return pair_df

### Контракт: `MaskingValidator`

**Зачем:** создать устойчивый offline-сигнал качества без platform `solution.csv`.  
**Вход:** preprocessed interactions.  
**Выход:** `observed`, `hidden_pairs`, `local_targets`.  
**Почему важно:** masking на уровне пары `(user_id, edition_id)` имитирует «потерянные позитивы».


In [4]:
class MaskingValidator:
    """Create local pseudo-incident split by hiding part of positive pairs.

    This simulates the competition setting and gives a deterministic
    offline signal for quick iteration.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def make_split(self, interactions: pd.DataFrame) -> dict[str, pd.DataFrame]:
        df = interactions.copy()
        max_day = df[COL_EVENT_TS].dt.normalize().max()
        incident_start = max_day - pd.Timedelta(days=self.cfg.incident_window_days - 1)

        incident_df = df[df[COL_EVENT_TS].dt.normalize() >= incident_start].copy()
        history_df = df[df[COL_EVENT_TS].dt.normalize() < incident_start].copy()

        incident_pairs = incident_df[[COL_USER, COL_ITEM]].drop_duplicates().reset_index(drop=True)
        if incident_pairs.empty:
            raise ValueError("Incident window has no pairs; cannot run masking validation")

        mask_count = max(1, int(len(incident_pairs) * self.cfg.mask_ratio))
        masked_pairs = incident_pairs.sample(n=mask_count, random_state=self.cfg.seed)
        masked_pairs = masked_pairs.assign(label=1)

        masked_key = set((int(r.user_id), int(r.edition_id)) for r in masked_pairs.itertuples(index=False))

        keep_mask = ~incident_df[[COL_USER, COL_ITEM]].apply(lambda x: (int(x.iloc[0]), int(x.iloc[1])) in masked_key, axis=1)
        observed_incident = incident_df[keep_mask].copy()
        observed_df = pd.concat([history_df, observed_incident], ignore_index=True)

        local_targets = masked_pairs[COL_USER].drop_duplicates().sort_values().to_frame()

        return {
            "observed_interactions": observed_df,
            "hidden_pairs": masked_pairs[[COL_USER, COL_ITEM, COL_LABEL]].copy(),
            "local_targets": local_targets,
            "history_interactions": history_df,
            "incident_interactions": incident_df,
        }

## Раздел кода 2 (Candidate Generation)

Ниже расположен код генерации кандидатов (`CandidateGenerator`) и логика coverage fallback.

### Контракт: `CandidateGenerator`

**Зачем:** прозрачный recall из трёх независимых источников, каждый легко расширять отдельно.  
**Вход:** `observed` и `target_users`.  
**Выход:** source-level кандидаты и merged candidate table с fallback покрытием.


In [5]:
class CandidateGenerator:
    """Generate recall candidates from simple, extensible baseline sources.

    Sources are intentionally transparent (popularity/content/i2i) so that
    participants can extend one source at a time without breaking contracts.
    """

    def __init__(self, cfg: Config, editions: pd.DataFrame, book_genres: pd.DataFrame):
        self.cfg = cfg
        self.editions = editions.copy()
        self.book_genres = book_genres.copy()

        self.edition_meta = self.editions[[
            COL_ITEM,
            COL_BOOK_ID,
            COL_AUTHOR_ID,
            COL_PUBLICATION_YEAR,
            COL_LANGUAGE_ID,
            COL_PUBLISHER_ID,
        ]].drop_duplicates()

        self.book_to_genres = (
            self.book_genres.groupby(COL_BOOK_ID)[COL_GENRE_ID].apply(lambda s: set(int(x) for x in s.dropna().tolist())).to_dict()
        )

    @staticmethod
    def _build_seen_map(interactions: pd.DataFrame) -> dict[int, set[int]]:
        seen = defaultdict(set)
        for row in interactions[[COL_USER, COL_ITEM]].drop_duplicates().itertuples(index=False):
            seen[int(row.user_id)].add(int(row.edition_id))
        return seen

    def global_recent_popularity(self, observed: pd.DataFrame, target_users: pd.DataFrame) -> pd.DataFrame:
        max_day = observed[COL_EVENT_TS].dt.normalize().max()
        all_rows = []
        seen_map = self._build_seen_map(observed)

        for w in POPULARITY_WINDOWS:
            start_day = max_day - pd.Timedelta(days=w - 1)
            win = observed[observed[COL_EVENT_TS].dt.normalize() >= start_day]
            pop = (
                win.groupby(COL_ITEM)[COL_USER]
                .nunique()
                .sort_values(ascending=False)
                .head(self.cfg.pop_top_n_per_window)
            )
            if pop.empty:
                continue
            top_items = [(int(ed), float(score)) for ed, score in pop.items()]
            source_name = f"pop_{w}d"
            users_iter = tqdm(
                target_users[COL_USER].tolist(),
                desc=f"pop candidates ({w}d)",
                leave=False,
            )
            for user_id in users_iter:
                user_id_int = int(user_id)
                user_seen = seen_map.get(user_id_int, set())
                for ed, score in top_items:
                    if ed in user_seen:
                        continue
                    all_rows.append((user_id_int, ed, source_name, score))

        out = pd.DataFrame(all_rows, columns=[COL_USER, COL_ITEM, COL_SOURCE, COL_SOURCE_SCORE])
        return out.drop_duplicates([COL_USER, COL_ITEM, COL_SOURCE]) if not out.empty else out

    def content_expansion(self, observed: pd.DataFrame, target_users: pd.DataFrame) -> pd.DataFrame:
        seen_map = self._build_seen_map(observed)
        meta = self.edition_meta.copy()

        user_recent = (
            observed.sort_values([COL_USER, COL_EVENT_TS], ascending=[True, False])
            .drop_duplicates([COL_USER, COL_ITEM]) 
            .groupby(COL_USER)
            .head(self.cfg.content_recent_k)
        )
        user_recent = user_recent.merge(meta, on=COL_ITEM, how=JOIN_LEFT)

        user_pref = {}
        pref_iter = tqdm(
            user_recent.groupby(COL_USER),
            desc="content profiles",
            leave=False,
        )
        for user_id, grp in pref_iter:
            author_pref = Counter(int(x) for x in grp[COL_AUTHOR_ID].dropna().tolist())
            lang_pref = Counter(int(x) for x in grp[COL_LANGUAGE_ID].dropna().tolist())
            year_vals = [int(x) for x in grp[COL_PUBLICATION_YEAR].dropna().tolist()]
            year_center = float(np.median(year_vals)) if year_vals else None
            user_pref[int(user_id)] = {
                "author": author_pref,
                "lang": lang_pref,
                "year_center": year_center,
            }

        candidates = []
        meta_rows = meta.to_dict(orient="records")
        users_iter = tqdm(
            target_users[COL_USER].tolist(),
            desc="content candidates",
            leave=False,
        )
        for user_id in users_iter:
            user_id = int(user_id)
            pref = user_pref.get(user_id)
            if not pref:
                continue
            seen = seen_map.get(user_id, set())

            scored = []
            for row in meta_rows:
                ed = int(row[COL_ITEM])
                if ed in seen:
                    continue
                score = 0.0
                author_id = row.get(COL_AUTHOR_ID)
                lang_id = row.get(COL_LANGUAGE_ID)
                pub_year = row.get(COL_PUBLICATION_YEAR)

                if pd.notna(author_id):
                    score += 2.5 * pref["author"].get(int(author_id), 0)
                if pd.notna(lang_id):
                    score += 1.0 * pref["lang"].get(int(lang_id), 0)
                if pref["year_center"] is not None and pd.notna(pub_year):
                    score += 1.0 / (1.0 + abs(float(pub_year) - pref["year_center"]))

                if score > 0:
                    scored.append((ed, score))

            scored.sort(key=lambda x: x[1], reverse=True)
            for ed, score in scored[: self.cfg.content_top_n]:
                candidates.append((user_id, ed, "content_hist", float(score)))

        out = pd.DataFrame(candidates, columns=[COL_USER, COL_ITEM, COL_SOURCE, COL_SOURCE_SCORE])
        return out.drop_duplicates([COL_USER, COL_ITEM, COL_SOURCE]) if not out.empty else out

    def i2i_cooccurrence(self, observed: pd.DataFrame, target_users: pd.DataFrame) -> pd.DataFrame:
        per_user_recent = (
            observed.sort_values([COL_USER, COL_EVENT_TS], ascending=[True, False])
            .drop_duplicates([COL_USER, COL_ITEM]) 
            .groupby(COL_USER)
            .head(self.cfg.i2i_recent_k_for_stats)
        )

        user_items = per_user_recent.groupby(COL_USER)[COL_ITEM].apply(lambda s: [int(x) for x in s.tolist()]).to_dict()

        item_cnt = Counter()
        pair_cnt = Counter()
        stats_iter = tqdm(user_items.values(), desc="i2i stats", leave=False)
        for items in stats_iter:
            unique_items = sorted(set(items))
            for i in unique_items:
                item_cnt[i] += 1
            for a, b in combinations(unique_items, 2):
                pair_cnt[(a, b)] += 1

        neigh = defaultdict(list)
        for (a, b), c in pair_cnt.items():
            sim = c / math.sqrt(item_cnt[a] * item_cnt[b])
            neigh[a].append((b, sim))
            neigh[b].append((a, sim))

        for item in list(neigh.keys()):
            neigh[item] = sorted(neigh[item], key=lambda x: x[1], reverse=True)[: self.cfg.i2i_neighbors_per_item]

        seen_map = self._build_seen_map(observed)
        per_user_anchors = (
            observed.sort_values([COL_USER, COL_EVENT_TS], ascending=[True, False])
            .drop_duplicates([COL_USER, COL_ITEM]) 
            .groupby(COL_USER)
            .head(self.cfg.i2i_anchor_k)
            .groupby(COL_USER)[COL_ITEM]
            .apply(lambda s: [int(x) for x in s.tolist()])
            .to_dict()
        )

        rows = []
        users_iter = tqdm(target_users[COL_USER].tolist(), desc="i2i candidates", leave=False)
        for user_id in users_iter:
            user_id = int(user_id)
            anchors = per_user_anchors.get(user_id, [])
            seen = seen_map.get(user_id, set())
            agg = defaultdict(float)
            for item in anchors:
                for nb, sim in neigh.get(item, []):
                    if nb in seen:
                        continue
                    agg[nb] += float(sim)
            ranked = sorted(agg.items(), key=lambda x: x[1], reverse=True)[: self.cfg.i2i_top_n]
            for ed, score in ranked:
                rows.append((user_id, int(ed), "i2i_cooc", float(score)))

        out = pd.DataFrame(rows, columns=[COL_USER, COL_ITEM, COL_SOURCE, COL_SOURCE_SCORE])
        return out.drop_duplicates([COL_USER, COL_ITEM, COL_SOURCE]) if not out.empty else out

    def merge_candidates(self, observed: pd.DataFrame, target_users: pd.DataFrame, parts: list[pd.DataFrame]) -> pd.DataFrame:
        non_empty = [df for df in parts if not df.empty]
        if not non_empty:
            raise ValueError("All candidate generators returned empty tables")

        merged = pd.concat(non_empty, ignore_index=True)
        merged = merged.drop_duplicates([COL_USER, COL_ITEM, COL_SOURCE]).copy()

        wide_flags = (
            merged.assign(flag=1)
            .pivot_table(index=[COL_USER, COL_ITEM], columns=COL_SOURCE, values="flag", aggfunc="max", fill_value=0)
            .reset_index()
        )

        wide_scores = (
            merged.pivot_table(index=[COL_USER, COL_ITEM], columns=COL_SOURCE, values=COL_SOURCE_SCORE, aggfunc="max", fill_value=0.0)
            .reset_index()
        )

        wide = wide_flags.merge(wide_scores, on=[COL_USER, COL_ITEM], how="outer", suffixes=("_flag", "_score")).fillna(0)

        rename_cols = {}
        for c in wide.columns:
            if c in {COL_USER, COL_ITEM}:
                continue
            if c.endswith("_flag"):
                rename_cols[c] = f"f_cand_from_{c.replace('_flag', '')}"
            elif c.endswith("_score"):
                rename_cols[c] = f"f_cand_score_{c.replace('_score', '')}"
        wide = wide.rename(columns=rename_cols)

        flag_cols = [c for c in wide.columns if c.startswith("f_cand_from_")]
        wide["f_candidate_source_count"] = wide[flag_cols].sum(axis=1) if flag_cols else 0

        seen_map = self._build_seen_map(observed)
        max_day = observed[COL_EVENT_TS].dt.normalize().max()
        pop30 = (
            observed[observed[COL_EVENT_TS].dt.normalize() >= max_day - pd.Timedelta(days=29)]
            .groupby(COL_ITEM)[COL_USER]
            .nunique()
            .sort_values(ascending=False)
            .index.tolist()
        )

        by_user = defaultdict(list)
        for row in wide[[COL_USER, COL_ITEM]].itertuples(index=False):
            by_user[int(row.user_id)].append(int(row.edition_id))

        fallback_rows = []
        users_iter = tqdm(target_users[COL_USER].tolist(), desc="coverage fallback", leave=False)
        for user_id in users_iter:
            user_id = int(user_id)
            current = set(by_user.get(user_id, []))
            seen = seen_map.get(user_id, set())
            for ed in pop30:
                ed = int(ed)
                if ed in current or ed in seen:
                    continue
                fallback_rows.append((user_id, ed))
                current.add(ed)
                if len(current) >= self.cfg.min_candidates_per_user:
                    break

        if fallback_rows:
            fallback = pd.DataFrame(fallback_rows, columns=[COL_USER, COL_ITEM])
            for col in wide.columns:
                if col not in {COL_USER, COL_ITEM}:
                    fallback[col] = 0.0
            wide = pd.concat([wide, fallback[wide.columns]], ignore_index=True)
            wide = wide.drop_duplicates([COL_USER, COL_ITEM]).copy()

        return wide

## Раздел кода 3 (Features + Ranking)

В этом блоке — только построение признаков и модель ранжирования:

- `FeatureBuilder`
- `BaselineRanker`
- сервисные функции для labels и локальной метрики

### Контракты: `FeatureBuilder`, `BaselineRanker`, `attach_labels`

- `FeatureBuilder`: строит групповые признаки (user/item/user-item/cooc/source).  
- `attach_labels`: присоединяет masking-истину к candidate rows.  
- `BaselineRanker`: обучает pointwise CatBoost и выдаёт score для сортировки top-k.


In [6]:
class FeatureBuilder:
    """Build compact tabular features for pointwise ranking baseline.

    Features are grouped by user/item/user-item/candidate-source to keep
    the notebook understandable and easy to evolve.
    """

    def __init__(self, cfg: Config, editions: pd.DataFrame, users: pd.DataFrame, book_genres: pd.DataFrame):
        self.cfg = cfg
        self.editions = editions.copy()
        self.users = users.copy()
        self.book_genres = book_genres.copy()

        self.meta = self.editions[[
            COL_ITEM,
            COL_BOOK_ID,
            COL_AUTHOR_ID,
            COL_PUBLICATION_YEAR,
            COL_AGE_RESTRICTION,
            COL_LANGUAGE_ID,
            COL_PUBLISHER_ID,
        ]].drop_duplicates()

        self.book_to_genres = (
            self.book_genres.groupby(COL_BOOK_ID)["genre_id"]
            .apply(lambda s: set(int(x) for x in s.dropna().tolist()))
            .to_dict()
        )

    def build(self, candidate_df: pd.DataFrame, observed: pd.DataFrame) -> pd.DataFrame:
        max_ts = observed[COL_EVENT_TS].max()

        feats = candidate_df.copy()
        feats = feats.merge(self.meta, on=COL_ITEM, how=JOIN_LEFT)
        feats = feats.merge(self.users[[COL_USER, COL_GENDER, COL_AGE]], on=COL_USER, how=JOIN_LEFT)

        # User features
        u_all = observed.groupby(COL_USER)
        u_last_ts = u_all[COL_EVENT_TS].max().rename("u_last_ts")
        u_cnt_180 = u_all.size().rename("f_user_cnt_events_180d")
        u_uniq_180 = u_all[COL_ITEM].nunique().rename("f_user_cnt_unique_editions_180d")
        u_cnt_read = observed[observed[COL_EVENT_TYPE] == 2].groupby(COL_USER).size().rename("f_user_cnt_read_180d")
        u_cnt_wish = observed[observed[COL_EVENT_TYPE] == 1].groupby(COL_USER).size().rename("f_user_cnt_wishlist_180d")

        last_30_start = max_ts.normalize() - pd.Timedelta(days=29)
        obs_30 = observed[observed[COL_EVENT_TS].dt.normalize() >= last_30_start]
        u_cnt_30 = obs_30.groupby(COL_USER).size().rename("f_user_cnt_events_30d")
        u_uniq_30 = obs_30.groupby(COL_USER)[COL_ITEM].nunique().rename("f_user_cnt_unique_editions_30d")

        user_feats = pd.concat([u_last_ts, u_cnt_180, u_uniq_180, u_cnt_read, u_cnt_wish, u_cnt_30, u_uniq_30], axis=1).fillna(0).reset_index()
        user_feats["f_user_days_since_last_event"] = (max_ts - user_feats["u_last_ts"]).dt.days
        user_feats["f_user_read_share_180d"] = user_feats["f_user_cnt_read_180d"] / user_feats["f_user_cnt_events_180d"].replace(0, np.nan)
        user_feats["f_user_read_share_180d"] = user_feats["f_user_read_share_180d"].fillna(0.0)

        feats = feats.merge(user_feats.drop(columns=["u_last_ts"]), on=COL_USER, how=JOIN_LEFT)

        feats["f_user_age"] = feats[COL_AGE].fillna(-1)
        feats["f_user_gender"] = feats[COL_GENDER].fillna(-1)
        feats["f_user_age_is_null"] = feats[COL_AGE].isna().astype(int)
        feats["f_user_gender_is_null"] = feats[COL_GENDER].isna().astype(int)

        # Item popularity features
        def pop_users(days: int, col_name: str) -> pd.Series:
            start = max_ts.normalize() - pd.Timedelta(days=days - 1)
            s = observed[observed[COL_EVENT_TS].dt.normalize() >= start].groupby(COL_ITEM)[COL_USER].nunique()
            return s.rename(col_name)

        def pop_events(days: int, col_name: str) -> pd.Series:
            start = max_ts.normalize() - pd.Timedelta(days=days - 1)
            s = observed[observed[COL_EVENT_TS].dt.normalize() >= start].groupby(COL_ITEM).size()
            return s.rename(col_name)

        i_pop_7 = pop_users(7, "f_item_pop_users_7d")
        i_pop_30 = pop_users(30, "f_item_pop_users_30d")
        i_pop_180 = pop_users(180, F_ITEM_POP_180D)
        i_events_30 = pop_events(30, "f_item_pop_events_30d")
        i_read_30 = observed[(observed[COL_EVENT_TYPE] == 2) & (observed[COL_EVENT_TS].dt.normalize() >= max_ts.normalize() - pd.Timedelta(days=29))].groupby(COL_ITEM).size().rename("f_item_pop_read_30d")
        i_wish_30 = observed[(observed[COL_EVENT_TYPE] == 1) & (observed[COL_EVENT_TS].dt.normalize() >= max_ts.normalize() - pd.Timedelta(days=29))].groupby(COL_ITEM).size().rename("f_item_pop_wishlist_30d")
        i_last_ts = observed.groupby(COL_ITEM)[COL_EVENT_TS].max().rename("i_last_ts")

        item_feats = pd.concat([i_pop_7, i_pop_30, i_pop_180, i_events_30, i_read_30, i_wish_30, i_last_ts], axis=1).fillna(0).reset_index()
        item_feats["f_item_days_since_last_seen"] = (max_ts - item_feats["i_last_ts"]).dt.days
        item_feats = item_feats.drop(columns=["i_last_ts"])

        author_pop = self.meta.merge(i_pop_180.reset_index(), on=COL_ITEM, how=JOIN_LEFT).fillna({F_ITEM_POP_180D: 0})
        author_pop = author_pop.groupby(COL_AUTHOR_ID)[F_ITEM_POP_180D].sum().rename("f_item_author_global_pop")
        book_pop = self.meta.merge(i_pop_180.reset_index(), on=COL_ITEM, how=JOIN_LEFT).fillna({F_ITEM_POP_180D: 0})
        book_pop = book_pop.groupby(COL_BOOK_ID)[F_ITEM_POP_180D].sum().rename("f_item_book_global_pop")

        feats = feats.merge(item_feats, on=COL_ITEM, how=JOIN_LEFT)
        feats = feats.merge(author_pop.reset_index(), on=COL_AUTHOR_ID, how=JOIN_LEFT)
        feats = feats.merge(book_pop.reset_index(), on=COL_BOOK_ID, how=JOIN_LEFT)

        # User-item count-based features
        obs_meta = observed.merge(self.meta, on=COL_ITEM, how=JOIN_LEFT)

        ua = obs_meta.groupby([COL_USER, COL_AUTHOR_ID]).size().rename("f_ui_author_count_in_user_history").reset_index()
        ub = obs_meta.groupby([COL_USER, COL_BOOK_ID]).size().rename("f_ui_book_count_in_user_history").reset_index()
        ul = obs_meta.groupby([COL_USER, COL_LANGUAGE_ID]).size().rename(F_UI_LANG_SEEN).reset_index()
        up = obs_meta.groupby([COL_USER, COL_PUBLISHER_ID]).size().rename(F_UI_PUB_SEEN).reset_index()

        recent_30 = obs_meta[obs_meta[COL_EVENT_TS].dt.normalize() >= max_ts.normalize() - pd.Timedelta(days=29)]
        ua30 = recent_30.groupby([COL_USER, COL_AUTHOR_ID]).size().rename("f_ui_recent_author_count_30d").reset_index()

        feats = feats.merge(ua, on=[COL_USER, COL_AUTHOR_ID], how=JOIN_LEFT)
        feats = feats.merge(ub, on=[COL_USER, COL_BOOK_ID], how=JOIN_LEFT)
        feats = feats.merge(ul, on=[COL_USER, COL_LANGUAGE_ID], how=JOIN_LEFT)
        feats = feats.merge(up, on=[COL_USER, COL_PUBLISHER_ID], how=JOIN_LEFT)
        feats = feats.merge(ua30, on=[COL_USER, COL_AUTHOR_ID], how=JOIN_LEFT)

        for c in [
            "f_ui_author_count_in_user_history",
            "f_ui_book_count_in_user_history",
            F_UI_LANG_SEEN,
            F_UI_PUB_SEEN,
            "f_ui_recent_author_count_30d",
        ]:
            feats[c] = feats[c].fillna(0)

        feats["f_ui_same_author_seen_before"] = (feats["f_ui_author_count_in_user_history"] > 0).astype(int)
        feats["f_ui_same_book_seen_before"] = (feats["f_ui_book_count_in_user_history"] > 0).astype(int)
        feats[F_UI_LANG_SEEN] = (feats[F_UI_LANG_SEEN] > 0).astype(int)
        feats[F_UI_PUB_SEEN] = (feats[F_UI_PUB_SEEN] > 0).astype(int)

        # Genre overlap + year proximity
        user_books = obs_meta.groupby(COL_USER)[COL_BOOK_ID].apply(lambda s: set(int(x) for x in s.dropna())).to_dict()
        user_genres = {}
        genre_iter = tqdm(user_books.items(), desc="user genre profiles", leave=False)
        for uid, books in genre_iter:
            gset = set()
            for b in books:
                gset |= self.book_to_genres.get(int(b), set())
            user_genres[int(uid)] = gset

        user_years = obs_meta.groupby(COL_USER)[COL_PUBLICATION_YEAR].apply(lambda s: [int(x) for x in s.dropna().tolist()]).to_dict()

        overlap_counts = []
        overlap_ratios = []
        min_year_diffs = []
        row_iter = tqdm(
            feats[[COL_USER, COL_BOOK_ID, COL_PUBLICATION_YEAR]].itertuples(index=False),
            total=len(feats),
            desc="user-item overlap features",
            leave=False,
        )
        for row in row_iter:
            uid = int(row.user_id)
            bid = int(row.book_id) if pd.notna(row.book_id) else None
            candidate_genres = self.book_to_genres.get(bid, set()) if bid is not None else set()
            ug = user_genres.get(uid, set())
            inter = candidate_genres & ug
            overlap_counts.append(len(inter))
            overlap_ratios.append(len(inter) / max(1, len(candidate_genres)))

            years = user_years.get(uid, [])
            if pd.notna(row.publication_year) and years:
                y = int(row.publication_year)
                min_year_diffs.append(min(abs(y - yy) for yy in years))
            else:
                min_year_diffs.append(999)

        feats["f_ui_genre_overlap_count"] = overlap_counts
        feats["f_ui_genre_overlap_ratio"] = overlap_ratios
        feats["f_ui_min_abs_year_diff_to_history"] = min_year_diffs

        # Cooc proxy features from candidate scores if present
        cooc_cols = [c for c in feats.columns if c.startswith("f_cand_score_i2i_cooc")]
        if cooc_cols:
            feats["f_cooc_max_sim_last10"] = feats[cooc_cols].max(axis=1)
            feats["f_cooc_sum_sim_last10"] = feats[cooc_cols].sum(axis=1)
            feats["f_cooc_mean_sim_last10"] = feats[cooc_cols].mean(axis=1)
            feats["f_cooc_hits_last10"] = (feats[cooc_cols] > 0).sum(axis=1)
        else:
            feats["f_cooc_max_sim_last10"] = 0.0
            feats["f_cooc_sum_sim_last10"] = 0.0
            feats["f_cooc_mean_sim_last10"] = 0.0
            feats["f_cooc_hits_last10"] = 0

        numeric_cols = [c for c in feats.columns if c.startswith("f_")]
        feats[numeric_cols] = feats[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        return feats

In [7]:
class BaselineRanker:
    """Train and apply CatBoost pointwise ranker for candidate pairs.

    Pointwise setup is used here for baseline clarity and robust debugging.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.model = CatBoostClassifier(
            iterations=cfg.catboost_iterations,
            depth=cfg.catboost_depth,
            learning_rate=cfg.catboost_learning_rate,
            l2_leaf_reg=cfg.catboost_l2_leaf_reg,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=cfg.seed,
            verbose=False,
        )
        self.feature_cols: list[str] = []

    def fit(self, train_df: pd.DataFrame, label_col: str = COL_LABEL) -> None:
        self.feature_cols = [c for c in train_df.columns if c.startswith("f_")]
        if not self.feature_cols:
            raise ValueError("No feature columns found for training")

        pos = float((train_df[label_col] == 1).sum())
        neg = float((train_df[label_col] == 0).sum())
        if pos == 0:
            raise ValueError("No positive labels in training set")
        class_weights = [1.0, max(1.0, neg / max(1.0, pos))]

        self.model.set_params(class_weights=class_weights)
        self.model.fit(train_df[self.feature_cols], train_df[label_col])

    def predict_scores(self, df: pd.DataFrame) -> np.ndarray:
        if not self.feature_cols:
            raise RuntimeError("Model is not fitted")
        return self.model.predict_proba(df[self.feature_cols])[:, 1]


class SubmissionBuilder:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def build_topk(
        self,
        scored_df: pd.DataFrame,
        target_users: pd.DataFrame,
        fallback_popularity: list[int],
        seen_map: dict[int, set[int]],
    ) -> pd.DataFrame:
        rows = []

        grouped = {
            int(uid): grp.sort_values(COL_SCORE, ascending=False)
            for uid, grp in scored_df.groupby(COL_USER)
        }

        users_iter = tqdm(target_users[COL_USER].tolist(), desc="build top-k", leave=False)
        for user_id in users_iter:
            user_id = int(user_id)
            picked = []
            used = set()

            if user_id in grouped:
                for ed in grouped[user_id][COL_ITEM].tolist():
                    ed = int(ed)
                    if ed in used:
                        continue
                    used.add(ed)
                    picked.append(ed)
                    if len(picked) == self.cfg.top_k:
                        break

            seen = seen_map.get(user_id, set())
            for ed in fallback_popularity:
                ed = int(ed)
                if ed in used or ed in seen:
                    continue
                used.add(ed)
                picked.append(ed)
                if len(picked) == self.cfg.top_k:
                    break

            if len(picked) < self.cfg.top_k:
                raise ValueError(f"User {user_id}: not enough items to fill top-{self.cfg.top_k}")

            for rank, ed in enumerate(picked, start=1):
                rows.append((user_id, ed, rank))

        return pd.DataFrame(rows, columns=[COL_USER, COL_ITEM, COL_RANK])


def attach_labels(feature_df: pd.DataFrame, hidden_pairs: pd.DataFrame) -> pd.DataFrame:
    """Attach binary labels from hidden pairs to candidate feature rows."""
    labels = hidden_pairs[[COL_USER, COL_ITEM]].copy()
    labels[COL_LABEL] = 1
    out = feature_df.merge(labels, on=[COL_USER, COL_ITEM], how=JOIN_LEFT)
    out[COL_LABEL] = out[COL_LABEL].fillna(0).astype(int)
    return out

## Шаг 1. Загрузка, preprocessing, masking

В этом блоке мы:

1. загружаем входные таблицы
2. делаем базовую очистку позитивных взаимодействий
3. формируем локальный masking-сценарий для оценки качества

`hidden_pairs` в этом сценарии играют роль локальной «правды», по которой считается `NDCG@20`

### Шаг 1.1 — Загрузка данных (checkpoint-aware)

**Зачем:** не читать CSV повторно при валидном cache.  
**Вход:** файлы в `data/`.  
**Выход:** `data` с таблицами baseline.


In [8]:
print_step("Load data")
loader = DataLoader(cfg)

raw_payload = {
    "required_files": {
        key: file_fingerprint(cfg.data_dir / name)
        for key, name in loader.REQUIRED_FILES.items()
    }
}
raw_hash = stage_expected_hash(STAGE_RAW_DATA, raw_payload)
raw_keys = [DATA_INTERACTIONS, DATA_TARGETS, DATA_EDITIONS, DATA_BOOK_GENRES, DATA_USERS]

raw_cached, _ = load_stage_df_dict(STAGE_RAW_DATA, raw_hash, raw_keys)
if raw_cached is not None:
    data = raw_cached
    checkpoint_status_print(STAGE_RAW_DATA, "CACHE HIT", f"tables={len(data)}")
else:
    data = loader.load()
    save_stage_df_dict(STAGE_RAW_DATA, data, raw_hash, raw_payload)
    checkpoint_status_print(STAGE_RAW_DATA, "CACHE SAVE", f"tables={len(data)}")

print({k: v.shape for k, v in data.items()})



============ Load data ============
[CACHE HIT] raw_data :: tables=5
{'interactions': (443278, 5), 'targets': (3862, 1), 'editions': (460599, 9), 'book_genres': (413301, 2), 'users': (80738, 3)}


### Шаг 1.2 — Preprocessing и masking (checkpoint-aware)

**Зачем:** не повторять очистку и masking после падения ноутбука.  
**Вход:** `data[interactions]` + параметры окна/seed/mask ratio.  
**Выход:** `interactions`, `pair_table`, `observed`, `hidden_pairs`, `local_targets`.


In [9]:
print_step("Preprocess interactions")
pre_payload = {
    "raw_hash": raw_hash,
    "history_window_days": cfg.history_window_days,
    "incident_window_days": cfg.incident_window_days,
    "mask_ratio": cfg.mask_ratio,
    "seed": cfg.seed,
}
pre_hash = stage_expected_hash(STAGE_PREPROCESS_MASKING, pre_payload)
pre_keys = ["interactions", "pair_table", "observed", "hidden_pairs", "local_targets"]

pre_cached, _ = load_stage_df_dict(STAGE_PREPROCESS_MASKING, pre_hash, pre_keys)
if pre_cached is not None:
    interactions = pre_cached["interactions"]
    pair_table = pre_cached["pair_table"]
    observed = pre_cached["observed"]
    hidden_pairs = pre_cached["hidden_pairs"]
    local_targets = pre_cached["local_targets"]
    checkpoint_status_print(STAGE_PREPROCESS_MASKING, "CACHE HIT", f"observed={observed.shape}")
else:
    pre = Preprocessor(cfg)
    interactions = pre.preprocess_interactions(data[DATA_INTERACTIONS])
    pair_table = pre.build_pair_table(interactions)

    print_step("Masking validation split")
    masker = MaskingValidator(cfg)
    split = masker.make_split(interactions)

    observed = split["observed_interactions"]
    hidden_pairs = split["hidden_pairs"]
    local_targets = split["local_targets"]

    save_stage_df_dict(
        STAGE_PREPROCESS_MASKING,
        {
            "interactions": interactions,
            "pair_table": pair_table,
            "observed": observed,
            "hidden_pairs": hidden_pairs,
            "local_targets": local_targets,
        },
        pre_hash,
        pre_payload,
    )
    checkpoint_status_print(STAGE_PREPROCESS_MASKING, "CACHE SAVE", f"observed={observed.shape}")

if observed.empty or hidden_pairs.empty or local_targets.empty:
    raise ValueError("Masking split produced empty tables; check data and window parameters")



============ Preprocess interactions ============
[CACHE HIT] preprocess_masking :: observed=(430353, 9)


### Шаг 1.3 — Sanity-check и мини-сводка

**Что проверяем:** размеры таблиц и примеры строк


In [10]:
print("interactions:", interactions.shape, "pair_table:", pair_table.shape)
print("observed:", observed.shape)
print("hidden_pairs:", hidden_pairs.shape)
print("local_targets:", local_targets.shape)

display(interactions.head(3))
display(pair_table.head(3))
display(hidden_pairs.head(5))


interactions: (443278, 9) pair_table: (442870, 8)
observed: (430353, 9)
hidden_pairs: (12909, 3)
local_targets: (4442, 1)


,user_id,edition_id,event_type,rating,event_ts,event_date,day_idx,is_incident_window,is_positive
0,150,1000956602,2,7.0,2025-05-01 11:39:29,2025-05-01,0,0,1
1,150,1013121397,1,NaN,2025-05-01 17:19:57,2025-05-01,0,0,1
2,150,1012702828,1,NaN,2025-05-01 17:23:04,2025-05-01,0,0,1


,user_id,edition_id,first_ts,last_ts,cnt_events,cnt_wishlist,cnt_read,mean_rating_read
0,150,1000102803,2025-09-29 12:57:50,2025-09-29 12:57:50,1,1,0,NaN
1,150,1000104617,2025-08-24 22:14:25,2025-08-24 22:14:25,1,0,1,NaN
2,150,1000112228,2025-06-14 23:17:29,2025-06-14 23:17:29,1,1,0,NaN


,user_id,edition_id,label
0,1621240,1002012225,1
1,163970,1001256821,1
2,12605200,1007473222,1
3,890850,1014593821,1
4,9738850,1005876634,1


## Шаг 2. Генерация кандидатов

Используем 3 простых источника:

- `Global Recent Popularity` (7/14/30 дней),
- `Content Expansion` по недавней истории пользователя,
- `Item-to-Item Co-occurrence`.

После этого объединяем источники, добавляем признаки происхождения кандидата и делаем coverage fallback.

### Шаг 2.1 — Генерация кандидатов с cache

**Зачем:** самые дорогие циклы (pop/content/i2i) не должны пересчитываться без необходимости.  
**Выход:** `c_pop`, `c_content`, `c_i2i`, `local_candidates`.


In [ ]:
print_step("Candidate generation (local validation targets)")

cand = CandidateGenerator(cfg, data[DATA_EDITIONS], data[DATA_BOOK_GENRES])

cand_payload = {
    "preprocess_hash": pre_hash,
    "target_users_hash": compute_hash(sorted(int(x) for x in local_targets[COL_USER].tolist())),
    "pop_top_n_per_window": cfg.pop_top_n_per_window,
    "content_top_n": cfg.content_top_n,
    "i2i_top_n": cfg.i2i_top_n,
    "content_recent_k": cfg.content_recent_k,
    "i2i_recent_k_for_stats": cfg.i2i_recent_k_for_stats,
    "i2i_anchor_k": cfg.i2i_anchor_k,
    "i2i_neighbors_per_item": cfg.i2i_neighbors_per_item,
    "min_candidates_per_user": cfg.min_candidates_per_user,
}
cand_hash = stage_expected_hash(STAGE_LOCAL_CANDIDATES, cand_payload)
cand_keys = ["c_pop", "c_content", "c_i2i", "local_candidates"]

cand_cached, _ = load_stage_df_dict(STAGE_LOCAL_CANDIDATES, cand_hash, cand_keys)
if cand_cached is not None:
    c_pop = cand_cached["c_pop"]
    c_content = cand_cached["c_content"]
    c_i2i = cand_cached["c_i2i"]
    local_candidates = cand_cached["local_candidates"]
    checkpoint_status_print(STAGE_LOCAL_CANDIDATES, "CACHE HIT", f"local_candidates={local_candidates.shape}")
else:
    c_pop = cand.global_recent_popularity(observed, local_targets)
    c_content = cand.content_expansion(observed, local_targets)
    c_i2i = cand.i2i_cooccurrence(observed, local_targets)

    local_candidates = cand.merge_candidates(observed, local_targets, [c_pop, c_content, c_i2i])
    save_stage_df_dict(
        STAGE_LOCAL_CANDIDATES,
        {
            "c_pop": c_pop,
            "c_content": c_content,
            "c_i2i": c_i2i,
            "local_candidates": local_candidates,
        },
        cand_hash,
        cand_payload,
    )
    checkpoint_status_print(STAGE_LOCAL_CANDIDATES, "CACHE SAVE", f"local_candidates={local_candidates.shape}")



============ Candidate generation (local validation targets) ============


pop candidates (7d):   0%|          | 0/4442 [00:00<?, ?it/s]

pop candidates (14d):   0%|          | 0/4442 [00:00<?, ?it/s]

pop candidates (30d):   0%|          | 0/4442 [00:00<?, ?it/s]

content profiles:   0%|          | 0/19072 [00:00<?, ?it/s]

content candidates:   0%|          | 0/4442 [00:00<?, ?it/s]

### Шаг 2.2 — Диагностика recall

In [ ]:
print("pop:", c_pop.shape, "content:", c_content.shape, "i2i:", c_i2i.shape)
print("merged local candidates:", local_candidates.shape)

display(local_candidates.head(5))

cand_counts = local_candidates.groupby(COL_USER)[COL_ITEM].nunique().rename("candidate_count")
print(cand_counts.describe().to_string())

plt.figure(figsize=(8, 4))
plt.hist(cand_counts.values, bins=25)
plt.title("Candidate count per user (local targets)")
plt.xlabel("Candidates per user")
plt.ylabel("Users")
plt.show()


## Шаг 3. Признаки и обучение CatBoost

На этом шаге:

1. строим табличные признаки для кандидатных пар
2. формируем бинарные labels из `hidden_pairs`
3. обучаем `CatBoostClassifier` в pointwise постановке
4. считаем локальный `NDCG@20` на masking-таргетах

## Раздел кода: Локальная метрика и контракт сабмита

Ниже объявлены утилиты, которые нужны сразу для двух задач:

- локальная оценка `NDCG@20` по masking-истине
- проверка корректности формата сабмита

In [ ]:
def ndcg_at_k(predicted: list[int], relevant: set[int], k: int = TOP_K) -> float:
    """Compute user-level NDCG@k for binary relevance."""
    dcg = 0.0
    for idx, edition_id in enumerate(predicted[:k], start=1):
        rel = 1.0 if int(edition_id) in relevant else 0.0
        dcg += rel / math.log2(idx + 1)
    idcg = sum(1.0 / math.log2(idx + 1) for idx in range(1, min(len(relevant), k) + 1))
    return dcg / idcg if idcg > 0.0 else 0.0


def evaluate_ndcg(submission_df: pd.DataFrame, hidden_pairs_df: pd.DataFrame, k: int = TOP_K) -> tuple[float, pd.DataFrame]:
    """Evaluate local masking quality using the same binary NDCG principle as task docs."""
    relevant_by_user: dict[int, set[int]] = defaultdict(set)
    for row in hidden_pairs_df[[COL_USER, COL_ITEM]].itertuples(index=False):
        relevant_by_user[int(row.user_id)].add(int(row.edition_id))

    pred_by_user: dict[int, list[int]] = {}
    for user_id, grp in submission_df.sort_values([COL_USER, COL_RANK]).groupby(COL_USER):
        pred_by_user[int(user_id)] = [int(x) for x in grp[COL_ITEM].tolist()[:k]]

    rows = []
    for user_id in sorted(relevant_by_user.keys()):
        ndcg = ndcg_at_k(pred_by_user.get(user_id, []), relevant_by_user[user_id], k=k)
        rows.append({COL_USER: user_id, "ndcg@20": ndcg, "hidden_relevant_count": len(relevant_by_user[user_id])})

    per_user = pd.DataFrame(rows)
    score = float(per_user["ndcg@20"].mean()) if not per_user.empty else 0.0
    return score, per_user


def validate_submission_rows_local(submission_rows: list[dict], target_users: set[str]) -> tuple[bool, list[str]]:
    """Validate baseline submission contract (20 unique items, ranks 1..20, all targets present)."""
    errors: list[str] = []
    by_user: dict[str, list[tuple[int, str]]] = defaultdict(list)

    for row in submission_rows:
        user_id = str(row.get(COL_USER, "")).strip()
        edition_id = str(row.get(COL_ITEM, "")).strip()
        rank_raw = str(row.get(COL_RANK, "")).strip()

        if not user_id or not edition_id or not rank_raw:
            errors.append("Rows must contain non-empty user_id, edition_id, rank")
            continue
        if not rank_raw.isdigit():
            errors.append(f"Rank must be integer for user {user_id}")
            continue

        rank = int(rank_raw)
        if rank < 1 or rank > TOP_K:
            errors.append(f"Rank out of range 1..{TOP_K} for user {user_id}: {rank}")

        by_user[user_id].append((rank, edition_id))

    missing = target_users - set(by_user.keys())
    extra = set(by_user.keys()) - target_users
    if missing:
        errors.append(f"Missing target users: {len(missing)}")
    if extra:
        errors.append(f"Unexpected users in submission: {len(extra)}")

    expected_ranks = set(range(1, TOP_K + 1))
    for user_id in sorted(by_user):
        rows = by_user[user_id]
        if len(rows) != TOP_K:
            errors.append(f"User {user_id}: expected {TOP_K} rows, got {len(rows)}")
            continue

        ranks = [rank for rank, _ in rows]
        editions = [edition_id for _, edition_id in rows]
        if len(set(ranks)) != TOP_K or set(ranks) != expected_ranks:
            errors.append(f"User {user_id}: ranks must be unique 1..{TOP_K}")
        if len(set(editions)) != TOP_K:
            errors.append(f"User {user_id}: edition_id must be unique")

    return len(errors) == 0, errors


def validate_submission_contract(submission_df: pd.DataFrame, target_user_ids: Iterable[int]) -> None:
    """Raise a clear exception if submission violates any required invariant."""
    rows = submission_df.to_dict(orient="records")
    target_users = {str(int(u)) for u in target_user_ids}
    ok, errors = validate_submission_rows_local(rows, target_users)
    if not ok:
        raise ValueError("; ".join(errors))

### Шаг 3.1 — Локальные фичи + train/inference с cache

**Зачем:** feature build + fit — самый дорогой участок после генерации кандидатов.  
**Выход:** `local_features`, `local_train`, `local_scored`, `local_submission`, `local_ndcg`.


In [ ]:
print_step("Feature building (local)")
feat_builder = FeatureBuilder(cfg, data[DATA_EDITIONS], data[DATA_USERS], data[DATA_BOOK_GENRES])

local_train_payload = {
    "candidates_hash": cand_hash,
    "catboost_iterations": cfg.catboost_iterations,
    "catboost_depth": cfg.catboost_depth,
    "catboost_learning_rate": cfg.catboost_learning_rate,
    "catboost_l2_leaf_reg": cfg.catboost_l2_leaf_reg,
    "top_k": cfg.top_k,
}
local_train_hash = stage_expected_hash(STAGE_LOCAL_FEATURES_TRAIN, local_train_payload)
local_train_keys = ["local_features", "local_train", "local_scored", "local_submission", "local_per_user"]

local_cached, _ = load_stage_df_dict(STAGE_LOCAL_FEATURES_TRAIN, local_train_hash, local_train_keys)
model_cached = load_stage_model(STAGE_LOCAL_FEATURES_TRAIN, local_train_hash)

if local_cached is not None and model_cached is not None:
    local_features = local_cached["local_features"]
    local_train = local_cached["local_train"]
    local_scored = local_cached["local_scored"]
    local_submission = local_cached["local_submission"]
    local_per_user = local_cached["local_per_user"]

    ranker = BaselineRanker(cfg)
    ranker.model.load_model(str(model_cached["model_path"]))
    ranker.feature_cols = model_cached["meta"]["feature_cols"]

    local_ndcg = float(read_json(stage_dir(STAGE_LOCAL_FEATURES_TRAIN) / CHECKPOINT_LOCAL_METRICS_FILE)["local_ndcg"])
    checkpoint_status_print(STAGE_LOCAL_FEATURES_TRAIN, "CACHE HIT", f"local_features={local_features.shape}")
else:
    local_features = feat_builder.build(local_candidates, observed)
    local_train = attach_labels(local_features, hidden_pairs)

    label_stats = local_train[COL_LABEL].value_counts().to_dict()
    print("local_features:", local_features.shape, "label stats:", label_stats)

    print_step("Train ranker (CatBoostClassifier)")
    ranker = BaselineRanker(cfg)
    ranker.fit(local_train, label_col=COL_LABEL)

    local_scored = local_train[[COL_USER, COL_ITEM]].copy()
    local_scored[COL_SCORE] = ranker.predict_scores(local_train)

    fallback_popularity = (
        observed.groupby(COL_ITEM)[COL_USER].nunique().sort_values(ascending=False).index.tolist()
    )
    seen_map_local = CandidateGenerator._build_seen_map(observed)

    sub_builder = SubmissionBuilder(cfg)
    local_submission = sub_builder.build_topk(local_scored, local_targets, fallback_popularity, seen_map_local)

    validate_submission_contract(local_submission, local_targets[COL_USER].tolist())
    local_ndcg, local_per_user = evaluate_ndcg(local_submission, hidden_pairs, k=cfg.top_k)

    save_stage_df_dict(
        STAGE_LOCAL_FEATURES_TRAIN,
        {
            "local_features": local_features,
            "local_train": local_train,
            "local_scored": local_scored,
            "local_submission": local_submission,
            "local_per_user": local_per_user,
        },
        local_train_hash,
        local_train_payload,
    )
    save_stage_model(STAGE_LOCAL_FEATURES_TRAIN, ranker, local_train_hash, local_train_payload)
    write_json(
        stage_dir(STAGE_LOCAL_FEATURES_TRAIN) / CHECKPOINT_LOCAL_METRICS_FILE,
        {"local_ndcg": float(local_ndcg)},
    )
    checkpoint_status_print(STAGE_LOCAL_FEATURES_TRAIN, "CACHE SAVE", f"local_features={local_features.shape}")

fallback_popularity = (
    observed.groupby(COL_ITEM)[COL_USER].nunique().sort_values(ascending=False).index.tolist()
)


### Шаг 3.2 — Локальная метрика

`NDCG@20` считается по скрытым `hidden_pairs` и показывает ранжирование baseline в offline-режиме.


In [ ]:
print(f"Local masking NDCG@{cfg.top_k}: {local_ndcg:.6f}")
display(local_per_user.head(5))


## Шаг 4. Финальный инференс и экспорт артефактов

В финале переиспользуем тот же pipeline для пользователей из `targets.csv`:

- генерируем кандидатов
- строим признаки
- получаем `score` через обученный CatBoost
- собираем `submission.csv` и валидируем формат
- сохраняем артефакты запуска

### Контракт: `SubmissionBuilder`

In [ ]:
class SubmissionBuilder:
    """Build final top-k predictions per user while preserving submission invariants.

    Why this class exists:
    - centralizes top-k dedup logic;
    - isolates fallback handling from model code;
    - makes contract checks easy to reuse in experiments.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def build_topk(
        self,
        scored_df: pd.DataFrame,
        target_users: pd.DataFrame,
        fallback_popularity: list[int],
        seen_map: dict[int, set[int]],
    ) -> pd.DataFrame:
        rows: list[tuple[int, int, int]] = []

        grouped = {
            int(uid): grp.sort_values(COL_SCORE, ascending=False)
            for uid, grp in scored_df.groupby(COL_USER)
        }

        users_iter = tqdm(target_users[COL_USER].tolist(), desc="build top-k", leave=False)
        for user_id in users_iter:
            user_id = int(user_id)
            picked: list[int] = []
            used: set[int] = set()

            # Primary source: model scores on generated candidates.
            if user_id in grouped:
                for ed in grouped[user_id][COL_ITEM].tolist():
                    ed = int(ed)
                    if ed in used:
                        continue
                    used.add(ed)
                    picked.append(ed)
                    if len(picked) == self.cfg.top_k:
                        break

            # Fallback guarantees strict top-k size for every target user.
            seen = seen_map.get(user_id, set())
            for ed in fallback_popularity:
                ed = int(ed)
                if ed in used or ed in seen:
                    continue
                used.add(ed)
                picked.append(ed)
                if len(picked) == self.cfg.top_k:
                    break

            if len(picked) < self.cfg.top_k:
                raise ValueError(f"User {user_id}: not enough items to fill top-{self.cfg.top_k}")

            for rank, ed in enumerate(picked, start=1):
                rows.append((user_id, ed, rank))

        return pd.DataFrame(rows, columns=[COL_USER, COL_ITEM, COL_RANK])

### Шаг 4.1 — Финальные кандидаты (checkpoint-aware)

**Зачем:** не пересчитывать full-target recall после падения ноутбука.  
**Выход:** `final_candidates` и source tables для final targets.


In [ ]:
print_step("Final inference for competition targets")
final_targets = data[DATA_TARGETS].copy()

final_cand_payload = {
    "preprocess_hash": pre_hash,
    "target_users_hash": compute_hash(sorted(int(x) for x in final_targets[COL_USER].tolist())),
    "pop_top_n_per_window": cfg.pop_top_n_per_window,
    "content_top_n": cfg.content_top_n,
    "i2i_top_n": cfg.i2i_top_n,
    "content_recent_k": cfg.content_recent_k,
    "i2i_recent_k_for_stats": cfg.i2i_recent_k_for_stats,
    "i2i_anchor_k": cfg.i2i_anchor_k,
    "i2i_neighbors_per_item": cfg.i2i_neighbors_per_item,
    "min_candidates_per_user": cfg.min_candidates_per_user,
}
final_cand_hash = stage_expected_hash(STAGE_FINAL_CANDIDATES, final_cand_payload)
final_cand_keys = ["c_pop_full", "c_content_full", "c_i2i_full", "final_candidates"]

final_cand_cached, _ = load_stage_df_dict(STAGE_FINAL_CANDIDATES, final_cand_hash, final_cand_keys)
if final_cand_cached is not None:
    c_pop_full = final_cand_cached["c_pop_full"]
    c_content_full = final_cand_cached["c_content_full"]
    c_i2i_full = final_cand_cached["c_i2i_full"]
    final_candidates = final_cand_cached["final_candidates"]
    checkpoint_status_print(STAGE_FINAL_CANDIDATES, "CACHE HIT", f"final_candidates={final_candidates.shape}")
else:
    c_pop_full = cand.global_recent_popularity(observed, final_targets)
    c_content_full = cand.content_expansion(observed, final_targets)
    c_i2i_full = cand.i2i_cooccurrence(observed, final_targets)

    final_candidates = cand.merge_candidates(observed, final_targets, [c_pop_full, c_content_full, c_i2i_full])
    save_stage_df_dict(
        STAGE_FINAL_CANDIDATES,
        {
            "c_pop_full": c_pop_full,
            "c_content_full": c_content_full,
            "c_i2i_full": c_i2i_full,
            "final_candidates": final_candidates,
        },
        final_cand_hash,
        final_cand_payload,
    )
    checkpoint_status_print(STAGE_FINAL_CANDIDATES, "CACHE SAVE", f"final_candidates={final_candidates.shape}")


### Шаг 4.2 — Финальные фичи, скоринг, экспорт артефактов

**Зачем:** из кандидатов собрать финальный сабмит и артефакты запуска (`submission`, `metrics`, `run_config`).


In [ ]:
final_stage_payload = {
    "final_candidates_hash": final_cand_hash,
    "local_train_hash": local_train_hash,
    "top_k": cfg.top_k,
}
final_stage_hash = stage_expected_hash(STAGE_FINAL_FEATURES_SCORES_SUBMISSION, final_stage_payload)
final_stage_keys = ["final_features", "final_scored"]

final_stage_cached, _ = load_stage_df_dict(STAGE_FINAL_FEATURES_SCORES_SUBMISSION, final_stage_hash, final_stage_keys)
artifact_submission = cfg.output_dir / ARTIFACT_SUBMISSION
artifact_metrics = cfg.output_dir / ARTIFACT_METRICS
artifact_config = cfg.output_dir / ARTIFACT_CONFIG

if final_stage_cached is not None and artifact_submission.exists() and artifact_metrics.exists() and artifact_config.exists():
    final_features = final_stage_cached["final_features"]
    final_scored = final_stage_cached["final_scored"]
    final_submission = pd.read_csv(artifact_submission)
    validate_submission_contract(final_submission, final_targets[COL_USER].tolist())
    checkpoint_status_print(STAGE_FINAL_FEATURES_SCORES_SUBMISSION, "CACHE HIT", f"submission_rows={len(final_submission)}")
else:
    final_features = feat_builder.build(final_candidates, observed)

    final_scored = final_features[[COL_USER, COL_ITEM]].copy()
    final_scored[COL_SCORE] = ranker.predict_scores(final_features)

    seen_map_full = CandidateGenerator._build_seen_map(observed)
    final_submission = sub_builder.build_topk(final_scored, final_targets, fallback_popularity, seen_map_full)
    validate_submission_contract(final_submission, final_targets[COL_USER].tolist())

    final_submission.to_csv(artifact_submission, index=False)
    final_submission.to_csv(cfg.root_submission_path, index=False)

    metrics = {
        "local_masking_ndcg@20": float(local_ndcg),
        "local_users": int(local_targets[COL_USER].nunique()),
        "targets_users": int(final_targets[COL_USER].nunique()),
        "local_candidates_avg_per_user": float(cand_counts.mean()),
        "local_candidates_min_per_user": int(cand_counts.min()),
    }

    write_json(artifact_metrics, metrics)
    payload = {k: str(v) if isinstance(v, Path) else v for k, v in asdict(cfg).items()}
    write_json(artifact_config, payload)

    save_stage_df_dict(
        STAGE_FINAL_FEATURES_SCORES_SUBMISSION,
        {
            "final_features": final_features,
            "final_scored": final_scored,
        },
        final_stage_hash,
        final_stage_payload,
    )
    write_json(
        stage_dir(STAGE_FINAL_FEATURES_SCORES_SUBMISSION) / "submission_meta.json",
        {
            "submission_rows": int(len(final_submission)),
            "submission_users": int(final_submission[COL_USER].nunique()),
            "contract": "OK",
            "expected_hash": final_stage_hash,
        },
    )
    checkpoint_status_print(STAGE_FINAL_FEATURES_SCORES_SUBMISSION, "CACHE SAVE", f"submission_rows={len(final_submission)}")

print("Saved:")
print("-", artifact_submission)
print("-", artifact_metrics)
print("-", artifact_config)
print("-", cfg.root_submission_path)
print("\nSubmission rows:", len(final_submission))
print("Users in submission:", final_submission[COL_USER].nunique())
print("Contract check: OK")

print_step("Checkpoint summary")
for row in checkpoint_summary:
    print(f"- {row['stage']}: {row['status']} ({row['details']})")

display(final_submission.head(10))
